# 30 — Choose windows and compute all metrics

Choose **fixed** or **detected** below, then Run All. No individual window review is required for either automatic policy. The selected policy is saved in the shared configuration for notebooks 40 and 50, and in every result sidecar.

**Fixed:** the first five seconds of each trace are noise, and the remaining record is the event data window. There is no one-second gap in this mode: the noise interval is exactly `[0,5)` and the event interval `[5,record end)` relative to each trace start.

**Detected:** use the trace-specific onset and audited noise prefix, with a one-second guard. The explicit fallback choice is fixed-five-second windows or skipping unresolved traces. Fallbacks/quality problems are flagged and not labelled manual reviews.

Event ends default to **record end**, not a detected coda end. Duration/energy features describe that selected record window. Counts remain uncalibrated and recorded clocks uncorrected. Complete metrics/features are cached per recording; switching policy preserves old versions and later reruns reuse unchanged computations.


Generated products use the shared `project_config.json` results root. Source/input files remain separate.


In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / 'project_config.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start in the Pinatubo repository; project_config.json must be available.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from pinatubo_project import input_path, output_path, runtime_environment
runtime_environment()

from pathlib import Path
import sys
_candidates = [Path.cwd(), Path.cwd() / '20_analysis', *Path.cwd().parents]
ANALYSIS_DIR = next((p for p in _candidates if (p / 'analysis_workflow.py').is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Start in 20_analysis or its project root, beside analysis_workflow.py.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
from analysis_workflow import load_context, describe_context
from IPython.display import display, Image, Markdown
import matplotlib.pyplot as plt
import pandas as pd
# The policy switch. Keep the full-archive file selection already in the JSON.
WINDOW_POLICY = 'fixed'       # 'fixed' or 'detected'; advanced manual mode is 'reviewed'
NOISE_SECONDS = 5.0
DETECTED_GUARD_SECONDS = 1.0
DETECTED_FALLBACK = 'fixed'   # 'fixed' or 'skip'
from analysis_workflow import configure_window_policy
configure_window_policy(ANALYSIS_DIR, policy=WINDOW_POLICY, noise_seconds=NOISE_SECONDS,
    detected_guard_s=DETECTED_GUARD_SECONDS, detected_fallback=DETECTED_FALLBACK)
ctx = load_context(ANALYSIS_DIR)
display(describe_context(ctx))


In [ ]:
from analysis_workflow import select_analysis_windows, compute_metrics, schema_table
selected, issues = select_analysis_windows(ctx)
display(selected.groupby(['window_policy', 'window_selection_method', 'window']).size().rename('windows').to_frame())
display(selected.head(20))
display(issues.head(30))
FORCE_RECOMPUTE = False
EXPORT_QUAKEML = True
metrics, errors = compute_metrics(ctx, force=FORCE_RECOMPUTE, write_quakeml=EXPORT_QUAKEML)
index = pd.read_csv(ctx['out'] / 'event_metrics_index.csv')
display(index.groupby('status').size().rename('recording_files').to_frame())
print('Selected windows:', len(selected), '; metric rows:', len(metrics), '; errors:', len(errors))
display(metrics.head(30))
display(errors.head(30))
print('Persistent sidecars:', ctx['products'] / 'event_metrics_cache')


## Full sidecars, full CSV and plotting view

Each EnhancedEvent sidecar preserves every output key: original EnhancedStream metrics, amplitude-spectrum arrays and summaries, SSAM, band ratios, station summaries, the complete FLOVOpy feature result (values, flags and provenance), matched PSD arrays/settings, window bounds and raw/processed artifact audits. Nothing is restricted to the plotting schema. Future keys are included automatically. Numeric arrays remain JSON arrays; nonfinite/missing values are JSON null. The original computed values remain available for audit even where the plotting view masks short-band measurements.

- `all_window_metrics.csv`: full dynamic trace/window export; arrays/lists are complete JSON-valued cells.
- `window_metrics.csv`: existing fixed-schema plotting/analysis view.
- `events_metadata.csv`: all event-sidecar metadata, including records without accepted windows.
- `event_metrics_index.csv`: exact active sidecar versions and cache status for this run.
- `catalog_summary.csv`: EnhancedCatalog's conventional event-summary view.
- `events.qml`: standard QuakeML; arbitrary waveform features remain in the sidecars, not QuakeML fields.

The stable cache is outside run-ID folders. Accepted window decisions also carry across selections by exact trace/source fingerprint; conflicting historical decisions are flagged. Changing review notes refreshes metadata without recalculating numeric features. Export-only regeneration below reads JSON, not waveforms.

Counts have no end-to-end calibration. All applicable count-domain EnhancedStream computations and shape/spectral features run for both accepted window types. Physical PGV/PGA/magnitude cannot be computed from unknown electronics. Recorded file starts are not fabricated earthquake origins. Metadata-only records therefore have no invented location, origin or magnitude.


The batch holds numerical arrays for one event at a time. Full CSV export scans sidecars twice to discover all keys and write rows; arrays remain complete JSON cells, while event metadata exports omit the redundant trace payload list. This avoids keeping all spectral arrays for the full archive in memory. Full numerical processing and array-rich exports can take substantially longer and more disk space than the earlier metadata-only run. You can interrupt and rerun; completed sidecars are reused.


In [ ]:
from analysis_workflow import export_event_metrics, load_event_metrics_catalog
REBUILD_EXPORTS_ONLY = False
if REBUILD_EXPORTS_ONLY:
    metrics = export_event_metrics(ctx, write_quakeml=EXPORT_QUAKEML)
# The generic FLOVOpy APIs are also available independently:
# cat = load_event_metrics_catalog(ctx)
# cat.export_metrics_csv('all_metrics.csv', level='trace')
# cat.export_metrics_csv('event_metadata.csv', level='event')
# cat.write('events.qml', format='QUAKEML')
print('Full metric CSV:', ctx['out'] / 'all_window_metrics.csv')
print('Active sidecar index:', ctx['out'] / 'event_metrics_index.csv')


In [ ]:
schema = schema_table()
schema.to_csv(ctx['out'] / 'metric_schema.csv', index=False)
display(schema)
from analysis_workflow import plot_metric_summary
fig = plot_metric_summary(ctx, metrics)
display(Image(filename=str(ctx['out'] / 'figures/accepted_window_metrics.png')))
plt.close(fig)
print('Provenance:', ctx['out'] / 'metric_provenance.json')


Run **40** for spectra/spectrograms and **50** for summaries after this notebook. They use the same saved window policy. Notebook 20 is optional for fixed mode and remains useful for inspecting detections. Quality flags still matter, but neither automatic mode requires manual acceptance. The separate `reviewed` policy is available for optional human-defined windows.
